[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YangKCLab/social-media-analysis/blob/main/docs/topics/data-collection/bluesky.ipynb)

# Bluesky API

Collect public data from Bluesky with the `atproto` Python SDK: search posts by keyword and time, look up an account's profile, and list its followers. The three sections share one login and are meant to be run in order.

**Setup.** Install `atproto` and `python-dotenv`, then put your handle and an
[app password](https://bsky.app/settings/app-passwords) in a `.env` file next
to the notebook:

```
bluesky_username=your-handle.bsky.social
bluesky_password=your-app-password
```

Never commit the `.env` file. In Google Colab there is no `.env` file, so set
the two values with `os.environ["bluesky_username"] = "..."` in a cell you
delete before sharing, or use Colab's Secrets panel.

Log in once per session, not once per request: logins are limited to 30 per
5 minutes and 300 per day. All endpoints together are limited to 3,000
requests per 5 minutes per IP address.

Reference: [Bluesky HTTP API](https://docs.bsky.app/docs/category/http-reference).

## Log in

In [ ]:
from atproto import Client
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv
import os

load_dotenv()

In [ ]:
client = Client()
client.login(os.getenv("bluesky_username"), os.getenv("bluesky_password"));

## Search posts

Search public posts by keyword with `app.bsky.feed.search_posts`, bound the search by time, and page through the results with the cursor.

In [ ]:
now = datetime.now(timezone.utc)
one_day_ago = now - timedelta(days=1)
iso_time_str = one_day_ago.isoformat().replace("+00:00", "Z")

In [ ]:
iso_time_str

In [ ]:
params = {
    "q": "cat",
    "limit": 10,
    "since": iso_time_str
}

In [ ]:
resps = client.app.bsky.feed.search_posts(
    params=params
)

In [ ]:
resps_dict = resps.model_dump()

In [ ]:
posts = resps_dict['posts']

In [ ]:
posts[0]

In [ ]:
resps_dict['cursor']

In [ ]:
params['cursor'] = resps_dict['cursor']

In [ ]:
resps_1 = client.app.bsky.feed.search_posts(
    params=params,
)

In [ ]:
resps_dict_1 = resps_1.model_dump()

In [ ]:
resps_dict_1['cursor']

## User profile

Look up one account's public profile with `app.bsky.actor.get_profile`: display name, description, follower and post counts, and the DID behind the handle.

In [ ]:
params = {
    "actor": "yang3kc.bsky.social"
}

In [ ]:
resps = client.app.bsky.actor.get_profile(
    params=params
)

In [ ]:
resps.model_dump()

## Followers

List the accounts that follow one account with `app.bsky.graph.get_followers`. The response is paginated the same way as post search: pass the cursor back to get the next page. `get_follows` returns the accounts the actor follows, with the same shape.

In [ ]:
params = {
    "actor": "yang3kc.bsky.social"
}

In [ ]:
resps = client.app.bsky.graph.get_followers(
    params=params
)

In [ ]:
resps.model_dump()